[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/078.%20The%20Fleet%20Sizing%20and%20Composition%20Problem/P78-Tier-5_executed.ipynb)

---


# 78. The Fleet Sizing and Composition Problem
## Tier 5: System-of-Systems Integration with a Digital-Twin Simulation

### Purpose and scope

This notebook builds a small, synthetic digital-twin simulation for a fleet serving urban and rural routes. It connects four activities: generating mock sensor readings, forecasting route demand, updating a virtual system state, and evaluating allocation scenarios. A simple threshold-based rule can add a small truck when a route's forecast service level falls below a specified threshold.

The example is an educational simulation, not a live digital twin or a fleet-sizing optimizer. It does not connect to physical vehicles, ingest real IoT data, estimate actual forecast accuracy, or prove that an allocation is globally optimal. Scenario metrics use the stated cost and capacity model; they should be interpreted within that model's limits.

### Inputs and outputs

- **Inputs:** three vehicle types, two routes, a baseline vehicle allocation, and synthetic historical demand and sensor readings.
- **Outputs:** simulated states, route service and cost metrics, scenario evaluations, plots, and a summary of model-derived insights.
- **Method:** a discrete-time simulation with a demand forecast, a service-threshold allocation heuristic, and what-if evaluation. The notebook does not use a mathematical programming solver.

### Model assumptions

- Each allocation entry is a nonnegative integer number of vehicles of one type assigned to one route.
- Vehicle capacity and route demand are measured in cargo units per planning day. The service-rate parameter is recorded but is not used because demand is not represented in compatible request-rate units.
- The reported cost combines one-time acquisition cost and one planning day's route operating cost. It is a modeled comparison measure, not a recurring daily cost or a discounted lifecycle cost.
- Forecasts and sensor readings are synthetic. Sensor accuracy is a declared input used in a synchronization-health proxy; it is not measured physical-to-virtual synchronization accuracy.
- The simulation advances one simulated minute per update. A one-second update interval is the modeled system setting; it is not a claim that the Python loop runs in one second.


## 1. Model and solution logic

An allocation matrix records how many vehicles of each type serve each route. For vehicle type \(i\) and route \(j\), let \(x_{ij}\) be the allocated vehicle count, \(q_i\) the cargo capacity of one vehicle, \(d_j\) the route demand forecast, \(a_i\) the acquisition cost, and \(c_{ij}\) the operating cost for one planning day on that route.

The capacity assigned to route \(j\) and its modeled service level are

$$C_j = \sum_i q_i x_{ij}, \qquad s_j = \min\left(1, \frac{C_j}{d_j}\right) \quad (d_j > 0).$$

If forecast demand is zero, the implementation defines the service level as 1. The model's combined cost is

$$K(x) = \sum_i \sum_j x_{ij}(a_i + c_{ij}).$$

The implementation reports the unweighted mean of route service levels. Its fleet-utilization metric is the fraction of total vehicle capacity that can be used to meet forecast demand:

$$U(x) = \frac{\sum_j \min(C_j, d_j)}{\sum_j C_j},$$

with utilization defined as zero when the fleet has no capacity. Thus, utilization is between zero and one; it is not the capacity-to-demand ratio.

### State transitions and the allocation rule

A **state** is the virtual system's record at one simulated time: allocation, vehicle and route conditions, sensor readings, forecast, and performance metrics. Advancing the state generates new synthetic readings and a new forecast, then recalculates the metrics. This repeated update is the simulation's central integration step.

At the scheduled decision points, the allocation rule checks each route. If its service level is below the threshold \(\tau = 0.9\), the rule adds one small truck to that route. In pseudocode, for each route \(j\):

$$x_{1j} \leftarrow x_{1j} + 1 \quad \text{if} \quad s_j < \tau.$$

This is a threshold heuristic, not an optimization recurrence: it does not search all allocations, minimize cost subject to constraints, or guarantee an optimal fleet. Its role is to demonstrate how a simple decision rule can be called within a state-update loop. With \(T\) simulation updates, \(V\) vehicle types, \(R\) routes, and \(S\) sensors, the dominant work here grows approximately as \(O(T(VR + S))\), excluding the fixed-size forecast-data generation.

In the code, `fleet_allocation` stores \(x\), `_calculate_performance_metrics` evaluates the cost and capacity equations, `update_state` advances the virtual state, and `simulate_optimization` implements the threshold rule. Scenario analysis evaluates changed allocations against the same metrics without modifying the running simulation state.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from time import perf_counter
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Digital-twin simulation environment initialized.")
print("Sensor readings and demand histories are synthetic.")

In [ ]:
@dataclass
class VehicleType:
    """Characteristics and modeled costs for one vehicle type."""

    id: int
    name: str
    capacity: int
    acquisition_cost: float
    operating_costs: Dict[int, float]
    service_rate: float
    fuel_efficiency: float
    maintenance_interval: int


@dataclass
class Route:
    """Demand and distance assumptions for one route."""

    id: int
    name: str
    demand_units: int
    arrival_rate: float
    max_delay_prob: float = 0.1
    distance: float = 100.0
    traffic_factor: float = 1.0


@dataclass
class IoTSensor:
    """A synthetic sensor definition and its declared characteristics."""

    id: int
    name: str
    type: str
    vehicle_id: Optional[int] = None
    route_id: Optional[int] = None
    update_frequency: float = 1.0
    accuracy: float = 0.95
    status: str = "active"


@dataclass
class DigitalTwinState:
    """The virtual system's recorded condition at one simulated time."""

    timestamp: datetime
    fleet_allocation: np.ndarray
    vehicle_states: Dict[int, Dict[str, Any]]
    route_conditions: Dict[int, Dict[str, Any]]
    sensor_data: Dict[int, Any]
    demand_forecast: np.ndarray
    performance_metrics: Dict[str, float]
    synchronization_health: float


@dataclass
class DigitalTwinResult:
    """Summary of a simulation; the final allocation is not an optimum."""

    final_fleet: Dict[Tuple[int, int], int]
    total_cost: float
    synchronization_accuracy: float
    prediction_accuracy: float
    scenario_improvements: Dict[str, float]
    computation_time: float
    system_uptime: float
    insights: List[str]

In [ ]:
class IoTSensorNetwork:
    """Generate mock sensor readings for vehicles and routes."""

    def __init__(self, vehicles: List[VehicleType], routes: List[Route]):
        self.vehicles = vehicles
        self.routes = routes
        self.sensors: List[IoTSensor] = []
        self._initialize_sensors()

    def _initialize_sensors(self) -> None:
        """Create four mock sensors per vehicle and one per route."""
        sensor_id = 1

        for vehicle in self.vehicles:
            for sensor_type, frequency, accuracy in [
                ("gps", 5.0, 0.98),
                ("fuel", 10.0, 0.95),
                ("load", 2.0, 0.92),
                ("maintenance", 60.0, 0.99),
            ]:
                self.sensors.append(
                    IoTSensor(
                        id=sensor_id,
                        name=f"{sensor_type.title()}_{vehicle.name}",
                        type=sensor_type,
                        vehicle_id=vehicle.id,
                        update_frequency=frequency,
                        accuracy=accuracy,
                    )
                )
                sensor_id += 1

        for route in self.routes:
            self.sensors.append(
                IoTSensor(
                    id=sensor_id,
                    name=f"Traffic_{route.name}",
                    type="traffic",
                    route_id=route.id,
                    update_frequency=30.0,
                    accuracy=0.90,
                )
            )
            sensor_id += 1

        print(f"Initialized {len(self.sensors)} synthetic sensors.")

    def generate_sensor_data(
        self, fleet_allocation: np.ndarray, timestamp: datetime
    ) -> Dict[int, Any]:
        """Return mock readings; allocation is accepted as simulation context."""
        del fleet_allocation
        readings: Dict[int, Any] = {}
        vehicles_by_id = {vehicle.id: vehicle for vehicle in self.vehicles}
        routes_by_id = {route.id: route for route in self.routes}

        for sensor in self.sensors:
            reading: Dict[str, Any] = {
                "timestamp": timestamp,
                "accuracy": sensor.accuracy,
            }

            if sensor.type == "gps":
                reading.update(
                    latitude=40.7128 + np.random.normal(0, 0.01),
                    longitude=-74.0060 + np.random.normal(0, 0.01),
                    speed=np.random.uniform(0, 80),
                )
            elif sensor.type == "fuel":
                reading.update(
                    fuel_level=np.random.uniform(20, 100),
                    consumption_rate=np.random.uniform(5, 15),
                    range_remaining=np.random.uniform(200, 800),
                )
            elif sensor.type == "load":
                vehicle = vehicles_by_id[sensor.vehicle_id]
                current_load = np.random.uniform(0, vehicle.capacity)
                reading.update(
                    current_load=current_load,
                    capacity_utilization=100 * current_load / vehicle.capacity,
                    load_cycles=int(np.random.randint(0, 50)),
                )
            elif sensor.type == "maintenance":
                reading.update(
                    engine_hours=np.random.uniform(0, 10000),
                    next_maintenance=int(np.random.randint(1, 30)),
                    health_score=np.random.uniform(70, 100),
                )
            elif sensor.type == "traffic":
                route = routes_by_id[sensor.route_id]
                reading.update(
                    route_name=route.name,
                    traffic_level=np.random.uniform(1, 10),
                    congestion_factor=np.random.uniform(1.0, 2.5),
                    average_speed=np.random.uniform(20, 80),
                    incidents=int(np.random.randint(0, 3)),
                )

            readings[sensor.id] = reading

        return readings

    def get_sensor_health(self) -> Dict[str, float]:
        """Summarize configured sensor status and declared accuracy."""
        active = [sensor for sensor in self.sensors if sensor.status == "active"]
        total = len(self.sensors)
        average_accuracy = (
            float(np.mean([sensor.accuracy for sensor in active])) if active else 0.0
        )
        return {
            "active_percentage": 100 * len(active) / total if total else 0.0,
            "average_accuracy": average_accuracy,
            "total_sensors": total,
            "active_sensors": len(active),
        }

In [ ]:
class PredictiveAnalytics:
    """Create synthetic demand histories, forecasts, and anomaly flags."""

    def __init__(self, routes: List[Route]):
        self.routes = routes
        self.historical_data = self._generate_historical_data()
        self.model_accuracy = 0.85

    def _generate_historical_data(self) -> Dict[int, np.ndarray]:
        """Generate one synthetic year of daily route demand."""
        days = 365
        day_index = np.arange(days)
        historical: Dict[int, np.ndarray] = {}

        for route in self.routes:
            base = route.demand_units
            trend = np.linspace(0, base * 0.2, days)
            seasonal = 10 * np.sin(2 * np.pi * day_index / 365)
            noise = np.random.normal(0, base * 0.1, days)
            historical[route.id] = np.maximum(base + trend + seasonal + noise, 1)

        return historical

    def forecast_demand(self, horizon_hours: int = 24) -> np.ndarray:
        """Forecast one demand value per route from recent synthetic history."""
        if horizon_hours <= 0:
            raise ValueError("horizon_hours must be positive")

        forecasts = np.zeros(len(self.routes), dtype=float)
        seasonal_factor = 1 + 0.1 * np.sin(
            2 * np.pi * datetime.now().timetuple().tm_yday / 365
        )

        for index, route in enumerate(self.routes):
            recent = self.historical_data[route.id][-30:]
            weekly_trend = np.mean(recent[-7:]) - np.mean(recent[-14:-7])
            base_forecast = np.mean(recent[-7:])
            forecast = base_forecast + weekly_trend * (horizon_hours / 24) * seasonal_factor
            uncertainty = np.random.normal(0, base_forecast * 0.1)
            forecasts[index] = max(1.0, forecast + uncertainty)

        return forecasts

    def detect_anomalies(self, current_demand: np.ndarray) -> List[Dict[str, Any]]:
        """Flag route demand more than two historical standard deviations away."""
        if len(current_demand) != len(self.routes):
            raise ValueError("current_demand must contain one value per route")

        anomalies: List[Dict[str, Any]] = []
        for index, route in enumerate(self.routes):
            historical = self.historical_data[route.id]
            mean_demand = float(np.mean(historical))
            std_demand = float(np.std(historical))
            deviation = abs(current_demand[index] - mean_demand)

            if deviation > 2 * std_demand:
                anomalies.append(
                    {
                        "route_id": route.id,
                        "route_name": route.name,
                        "current_demand": float(current_demand[index]),
                        "expected_range": (
                            mean_demand - 2 * std_demand,
                            mean_demand + 2 * std_demand,
                        ),
                        "severity": "high" if deviation > 3 * std_demand else "medium",
                    }
                )

        return anomalies

    def get_prediction_confidence(self) -> float:
        """Return a simulated confidence proxy, not measured forecast accuracy."""
        return float(np.clip(self.model_accuracy + np.random.normal(0, 0.05), 0, 1))


print("Synthetic forecasting and anomaly-detection components are ready.")

In [ ]:
class IntegratedDigitalTwin:
    """Coordinate synthetic sensing, forecasting, state updates, and metrics."""

    def __init__(self, vehicles: List[VehicleType], routes: List[Route]):
        self.vehicles = vehicles
        self.routes = routes
        self.sensor_network = IoTSensorNetwork(vehicles, routes)
        self.analytics = PredictiveAnalytics(routes)
        self.current_state: Optional[DigitalTwinState] = None
        self.state_history: List[DigitalTwinState] = []
        self.sync_accuracy_history: List[float] = []
        self.prediction_accuracy_history: List[float] = []
        self.system_uptime = 100.0
        self.update_frequency = 1.0
        self.simulation_speed = 60

    def _validate_allocation(self, allocation: np.ndarray) -> np.ndarray:
        """Enforce the dimensions and integer-count meaning of an allocation."""
        allocation = np.asarray(allocation)
        expected_shape = (len(self.vehicles), len(self.routes))
        if allocation.shape != expected_shape:
            raise ValueError(f"Allocation must have shape {expected_shape}")
        if not np.issubdtype(allocation.dtype, np.number):
            raise ValueError("Allocation entries must be numeric vehicle counts")
        if not np.all(np.isfinite(allocation)):
            raise ValueError("Allocation entries must be finite")
        if np.any(allocation < 0) or not np.all(allocation == np.floor(allocation)):
            raise ValueError("Allocation entries must be nonnegative integers")
        return allocation.astype(int, copy=True)

    def _generate_vehicle_states(
        self, fleet_allocation: np.ndarray
    ) -> Dict[int, Dict[str, Any]]:
        """Create synthetic state records for each vehicle type."""
        states: Dict[int, Dict[str, Any]] = {}
        for index, vehicle in enumerate(self.vehicles):
            assigned = int(np.sum(fleet_allocation[index]))
            states[vehicle.id] = {
                "name": vehicle.name,
                "total_assigned": assigned,
                "status": "active" if assigned > 0 else "idle",
                "fuel_level": float(np.random.uniform(20, 100)),
                "maintenance_hours": float(np.random.uniform(0, 500)),
                "efficiency": float(np.random.uniform(0.85, 1.0)),
                "last_maintenance": datetime.now()
                - timedelta(days=int(np.random.randint(0, 30))),
            }
        return states

    def _generate_route_conditions(self) -> Dict[int, Dict[str, Any]]:
        """Create synthetic traffic and road-condition records."""
        return {
            route.id: {
                "name": route.name,
                "traffic_level": float(np.random.uniform(1, 10)),
                "weather_factor": float(np.random.uniform(0.8, 1.2)),
                "road_condition": str(np.random.choice(["good", "fair", "poor"])),
                "congestion_factor": float(np.random.uniform(1.0, 2.0)),
                "incidents": int(np.random.randint(0, 3)),
            }
            for route in self.routes
        }

    def _calculate_performance_metrics(
        self, fleet_allocation: np.ndarray, demand_forecast: np.ndarray
    ) -> Dict[str, float]:
        """Evaluate the stated cost, route service, and capacity-use equations."""
        allocation = self._validate_allocation(fleet_allocation)
        demand = np.asarray(demand_forecast, dtype=float)
        if demand.shape != (len(self.routes),) or np.any(demand < 0):
            raise ValueError("Demand forecast must be nonnegative with one value per route")

        total_cost = 0.0
        total_capacity = 0.0
        route_capacities = np.zeros(len(self.routes), dtype=float)

        for vehicle_index, vehicle in enumerate(self.vehicles):
            for route_index, route in enumerate(self.routes):
                count = allocation[vehicle_index, route_index]
                total_cost += count * (
                    vehicle.acquisition_cost + vehicle.operating_costs[route.id]
                )
                capacity = count * vehicle.capacity
                route_capacities[route_index] += capacity
                total_capacity += capacity

        service_levels = [
            min(1.0, capacity / route_demand) if route_demand > 0 else 1.0
            for capacity, route_demand in zip(route_capacities, demand)
        ]
        served_capacity = sum(
            min(capacity, route_demand)
            for capacity, route_demand in zip(route_capacities, demand)
        )
        utilization = served_capacity / total_capacity if total_capacity > 0 else 0.0

        return {
            "total_cost": float(total_cost),
            "total_capacity": float(total_capacity),
            "average_service_level": float(np.mean(service_levels)),
            "fleet_utilization": float(utilization),
            "cost_efficiency": float(total_cost / total_capacity)
            if total_capacity > 0
            else 0.0,
            "min_service_level": float(min(service_levels)),
            "max_service_level": float(max(service_levels)),
        }

    def initialize_state(self, fleet_allocation: np.ndarray) -> DigitalTwinState:
        """Create and store the initial synthetic state for an allocation."""
        allocation = self._validate_allocation(fleet_allocation)
        timestamp = datetime.now()
        sensor_data = self.sensor_network.generate_sensor_data(allocation, timestamp)
        demand_forecast = self.analytics.forecast_demand()
        metrics = self._calculate_performance_metrics(allocation, demand_forecast)
        state = DigitalTwinState(
            timestamp=timestamp,
            fleet_allocation=allocation,
            vehicle_states=self._generate_vehicle_states(allocation),
            route_conditions=self._generate_route_conditions(),
            sensor_data=sensor_data,
            demand_forecast=demand_forecast,
            performance_metrics=metrics,
            synchronization_health=self._calculate_synchronization_health(sensor_data),
        )
        self.current_state = state
        self.state_history.append(state)
        return state

    def _calculate_synchronization_health(self, sensor_data: Dict[int, Any]) -> float:
        """Combine mock-reading coverage and declared sensor accuracy as a proxy."""
        total_sensors = len(self.sensor_network.sensors)
        if total_sensors == 0:
            return 0.0
        coverage = len(sensor_data) / total_sensors
        accuracies = [
            reading["accuracy"]
            for reading in sensor_data.values()
            if "accuracy" in reading
        ]
        average_accuracy = float(np.mean(accuracies)) if accuracies else 0.0
        return float(np.clip(coverage * average_accuracy, 0.0, 1.0))

    def _update_vehicle_states(
        self, fleet_allocation: np.ndarray
    ) -> Dict[int, Dict[str, Any]]:
        """Update mock vehicle status while carrying forward gradual changes."""
        if self.current_state is None:
            return self._generate_vehicle_states(fleet_allocation)

        states = {
            vehicle_id: state.copy()
            for vehicle_id, state in self.current_state.vehicle_states.items()
        }
        for index, vehicle in enumerate(self.vehicles):
            assigned = int(np.sum(fleet_allocation[index]))
            previous = states[vehicle.id]
            states[vehicle.id] = {
                "name": vehicle.name,
                "total_assigned": assigned,
                "status": "active" if assigned > 0 else "idle",
                "fuel_level": max(0.0, previous["fuel_level"] - np.random.uniform(0, 2)),
                "maintenance_hours": previous["maintenance_hours"]
                + np.random.uniform(0, 1),
                "efficiency": max(
                    0.7, previous["efficiency"] - np.random.uniform(0, 0.02)
                ),
                "last_maintenance": previous["last_maintenance"],
            }
        return states

    def _update_route_conditions(self) -> Dict[int, Dict[str, Any]]:
        """Apply small synthetic changes to the recorded route conditions."""
        if self.current_state is None:
            return self._generate_route_conditions()

        conditions = {
            route_id: value.copy()
            for route_id, value in self.current_state.route_conditions.items()
        }
        for route_id, previous in conditions.items():
            conditions[route_id] = {
                "name": previous["name"],
                "traffic_level": float(
                    np.clip(previous["traffic_level"] + np.random.normal(0, 0.5), 1, 10)
                ),
                "weather_factor": float(
                    np.clip(previous["weather_factor"] + np.random.normal(0, 0.1), 0.5, 1.5)
                ),
                "road_condition": previous["road_condition"],
                "congestion_factor": float(
                    np.clip(previous["congestion_factor"] + np.random.normal(0, 0.2), 1, 3)
                ),
                "incidents": max(
                    0,
                    previous["incidents"]
                    + int(np.random.choice([-1, 0, 1], p=[0.1, 0.8, 0.1])),
                ),
            }
        return conditions

    def update_state(
        self, new_allocation: Optional[np.ndarray] = None
    ) -> DigitalTwinState:
        """Advance one simulated step and record the resulting state."""
        if self.current_state is None:
            raise ValueError("Initialize the digital-twin state before updating it")

        allocation = (
            self.current_state.fleet_allocation.copy()
            if new_allocation is None
            else self._validate_allocation(new_allocation)
        )
        elapsed_seconds = self.update_frequency * self.simulation_speed
        timestamp = self.current_state.timestamp + timedelta(seconds=elapsed_seconds)
        sensor_data = self.sensor_network.generate_sensor_data(allocation, timestamp)
        demand_forecast = self.analytics.forecast_demand()
        metrics = self._calculate_performance_metrics(allocation, demand_forecast)
        state = DigitalTwinState(
            timestamp=timestamp,
            fleet_allocation=allocation,
            vehicle_states=self._update_vehicle_states(allocation),
            route_conditions=self._update_route_conditions(),
            sensor_data=sensor_data,
            demand_forecast=demand_forecast,
            performance_metrics=metrics,
            synchronization_health=self._calculate_synchronization_health(sensor_data),
        )
        self.current_state = state
        self.state_history.append(state)
        self.sync_accuracy_history.append(state.synchronization_health)
        return state

In [ ]:
vehicles = [
    VehicleType(
        id=1,
        name="Small Truck",
        capacity=5,
        acquisition_cost=50000,
        operating_costs={1: 1000, 2: 1500},
        service_rate=4.0,
        fuel_efficiency=8.0,
        maintenance_interval=30,
    ),
    VehicleType(
        id=2,
        name="Medium Truck",
        capacity=10,
        acquisition_cost=80000,
        operating_costs={1: 1200, 2: 1800},
        service_rate=3.5,
        fuel_efficiency=6.0,
        maintenance_interval=45,
    ),
    VehicleType(
        id=3,
        name="Large Truck",
        capacity=20,
        acquisition_cost=120000,
        operating_costs={1: 1500, 2: 2000},
        service_rate=3.0,
        fuel_efficiency=4.0,
        maintenance_interval=60,
    ),
]

routes = [
    Route(
        id=1,
        name="Urban Route",
        demand_units=15,
        arrival_rate=8.0,
        max_delay_prob=0.1,
        distance=80.0,
        traffic_factor=1.5,
    ),
    Route(
        id=2,
        name="Rural Route",
        demand_units=25,
        arrival_rate=4.0,
        max_delay_prob=0.1,
        distance=150.0,
        traffic_factor=1.0,
    ),
]

print("FLEET-SIZING INSTANCE")
print("=" * 60)
for vehicle in vehicles:
    print(
        f"{vehicle.name}: capacity {vehicle.capacity} cargo units, "
        f"acquisition cost ${vehicle.acquisition_cost:,.0f}, "
        f"fuel efficiency {vehicle.fuel_efficiency:.1f} km/L"
    )
print()
for route in routes:
    print(
        f"{route.name}: demand {route.demand_units} cargo units per planning day, "
        f"distance {route.distance:.1f} km, traffic factor {route.traffic_factor:.1f}"
    )

## 2. Initialize the virtual system

The baseline allocation below is a starting configuration, not a claimed optimum. The setup creates the sensor and analytics components, then records an initial state. Its service levels compare each route's assigned capacity with that route's own forecast; capacity assigned to one route does not automatically serve another route.

In [ ]:
digital_twin = IntegratedDigitalTwin(vehicles, routes)

initial_allocation = np.array(
    [
        [2, 1],  # Small trucks: two on Urban, one on Rural
        [1, 2],  # Medium trucks: one on Urban, two on Rural
        [0, 1],  # Large trucks: none on Urban, one on Rural
    ],
    dtype=int,
)

initial_state = digital_twin.initialize_state(initial_allocation)
sensor_health = digital_twin.sensor_network.get_sensor_health()

print("INITIAL SIMULATED STATE")
print("=" * 60)
print(f"Timestamp: {initial_state.timestamp:%Y-%m-%d %H:%M:%S}")
print("Allocation matrix (vehicle types by routes):")
print(initial_state.fleet_allocation)
print(f"Combined modeled cost: ${initial_state.performance_metrics['total_cost']:,.2f}")
print(
    "Mean route service level: "
    f"{initial_state.performance_metrics['average_service_level']:.1%}"
)
print(
    "Synchronization-health proxy: "
    f"{initial_state.synchronization_health:.1%}"
)
print(
    "Synthetic demand forecast (cargo units): "
    f"{np.round(initial_state.demand_forecast, 1).tolist()}"
)
print(
    f"Active sensors: {sensor_health['active_sensors']}/"
    f"{sensor_health['total_sensors']} "
    f"({sensor_health['active_percentage']:.1f}%)"
)
print(f"Mean declared sensor accuracy: {sensor_health['average_accuracy']:.1%}")

In [ ]:
def simulate_optimization(
    current_state: DigitalTwinState,
    digital_twin: IntegratedDigitalTwin,
    service_threshold: float = 0.9,
) -> Optional[np.ndarray]:
    """Add one small truck to each route below the service threshold."""
    if not 0 <= service_threshold <= 1:
        raise ValueError("service_threshold must be between zero and one")

    allocation = current_state.fleet_allocation.copy()
    metrics = digital_twin._calculate_performance_metrics(
        allocation, current_state.demand_forecast
    )

    if metrics["min_service_level"] >= service_threshold:
        return None

    route_capacities = np.zeros(len(digital_twin.routes), dtype=float)
    for vehicle_index, vehicle in enumerate(digital_twin.vehicles):
        route_capacities += allocation[vehicle_index] * vehicle.capacity

    for route_index, demand in enumerate(current_state.demand_forecast):
        service_level = min(1.0, route_capacities[route_index] / demand) if demand > 0 else 1.0
        if service_level < service_threshold:
            allocation[0, route_index] += 1

    return allocation


def run_simulation(
    digital_twin: IntegratedDigitalTwin, duration_hours: int = 24
) -> List[DigitalTwinState]:
    """Advance the twin in simulated one-minute steps for the requested horizon."""
    if duration_hours <= 0:
        raise ValueError("duration_hours must be positive")
    if digital_twin.current_state is None:
        raise ValueError("Initialize the digital-twin state before simulation")

    total_updates = int(
        duration_hours
        * 3600
        / (digital_twin.update_frequency * digital_twin.simulation_speed)
    )
    simulation_states: List[DigitalTwinState] = []
    anomaly_count = 0
    optimization_count = 0

    print("STARTING SYNTHETIC DIGITAL-TWIN SIMULATION")
    print(f"Simulated horizon: {duration_hours} hours")
    print(
        "Simulated time per update: "
        f"{digital_twin.update_frequency * digital_twin.simulation_speed:.0f} seconds"
    )
    print(f"Number of updates: {total_updates}")

    for update in range(total_updates):
        current_state = digital_twin.update_state()

        if update % 10 == 0:
            reference_demand = np.array(
                [route.demand_units for route in digital_twin.routes], dtype=float
            )
            anomaly_count += len(digital_twin.analytics.detect_anomalies(reference_demand))

        if update > 0 and update % 50 == 0:
            new_allocation = simulate_optimization(current_state, digital_twin)
            if new_allocation is not None:
                current_state = digital_twin.update_state(new_allocation)
                optimization_count += 1

        simulation_states.append(current_state)

        progress_interval = max(1, total_updates // 10)
        if update % progress_interval == 0:
            progress = 100 * (update + 1) / total_updates
            service = current_state.performance_metrics["average_service_level"]
            print(
                f"Update {update + 1:4d}/{total_updates} "
                f"({progress:5.1f}%): mean service {service:.1%}, "
                f"sync proxy {current_state.synchronization_health:.1%}"
            )

    print("Simulation loop completed.")
    print(f"Recorded loop states: {len(simulation_states)}")
    print(f"Threshold-rule interventions: {optimization_count}")
    print(f"Anomaly flags from synthetic reference checks: {anomaly_count}")
    return simulation_states


simulation_states = run_simulation(digital_twin, duration_hours=24)

## 3. Evaluate what-if scenarios

A what-if analysis changes one modeled input or allocation and recomputes the same metrics. It is useful only when the scenario change is represented in the model. The fuel-price scenario below therefore evaluates a fuel-efficiency-oriented allocation shift under the existing cost formula; it cannot estimate the effect of a fuel-price shock because the cost model contains no fuel-price parameter. The increased-demand scenario explicitly scales the forecast used for its service calculation.

In [ ]:
def analyze_scenarios(
    digital_twin: IntegratedDigitalTwin,
) -> Dict[str, Dict[str, Any]]:
    """Evaluate allocation scenarios without changing the running twin state."""
    if digital_twin.current_state is None:
        raise ValueError("Initialize the digital-twin state before scenario analysis")

    baseline = digital_twin.current_state.fleet_allocation.copy()
    baseline_demand = digital_twin.current_state.demand_forecast.copy()
    scenarios: Dict[str, Tuple[np.ndarray, np.ndarray, str]] = {
        "baseline": (
            baseline.copy(),
            baseline_demand.copy(),
            "Current allocation and current synthetic forecast.",
        ),
        "increased_demand": (
            baseline.copy(),
            baseline_demand * 1.25,
            "Forecast demand increased by 25%; allocation held fixed.",
        ),
        "fuel_price_shock": (
            baseline.copy(),
            baseline_demand.copy(),
            "Allocation shift only; fuel-price effects are not modeled.",
        ),
        "maintenance_crisis": (
            baseline.copy(),
            baseline_demand.copy(),
            "One available large truck is replaced by a medium truck on its route.",
        ),
        "traffic_congestion": (
            baseline.copy(),
            baseline_demand.copy(),
            "One urban medium truck is reassigned to the small-truck type.",
        ),
    }

    fuel_allocation = scenarios["fuel_price_shock"][0]
    for route_index in range(len(digital_twin.routes)):
        if fuel_allocation[2, route_index] > 0:
            fuel_allocation[2, route_index] -= 1
            fuel_allocation[0, route_index] += 1

    maintenance_allocation = scenarios["maintenance_crisis"][0]
        
    large_vehicle_index = 2
    medium_vehicle_index = 1
    for route_index in range(len(digital_twin.routes)):
        if maintenance_allocation[large_vehicle_index, route_index] > 0:
            maintenance_allocation[large_vehicle_index, route_index] -= 1
            maintenance_allocation[medium_vehicle_index, route_index] += 1
            break

    traffic_allocation = scenarios["traffic_congestion"][0]
    urban_route_index = next(
        index for index, route in enumerate(digital_twin.routes) if route.id == 1
    )
    if traffic_allocation[1, urban_route_index] > 0:
        traffic_allocation[1, urban_route_index] -= 1
        traffic_allocation[0, urban_route_index] += 1

    results: Dict[str, Dict[str, Any]] = {}
    print("WHAT-IF SCENARIO ANALYSIS")
    print("=" * 60)

    for name, (allocation, demand, interpretation) in scenarios.items():
        metrics = digital_twin._calculate_performance_metrics(allocation, demand)
        results[name] = {
            "allocation": allocation.copy(),
            "demand": demand.copy(),
            "interpretation": interpretation,
            **metrics,
        }
        print(f"\nScenario: {name.replace('_', ' ').title()}")
        print(f"  Interpretation: {interpretation}")
        print(f"  Combined modeled cost: ${metrics['total_cost']:,.2f}")
        print(f"  Mean route service level: {metrics['average_service_level']:.1%}")
        print(f"  Fleet utilization: {metrics['fleet_utilization']:.1%}")

    return results


scenario_results = analyze_scenarios(digital_twin)

## 4. Visualize simulated results

The plots summarize recorded synthetic states and scenario metrics. The demand plot compares forecasts with configured route-demand values; those configured values are reference inputs, not observed outcomes. Consequently, the plot is not a measurement of forecast accuracy.

In [ ]:
def visualize_digital_twin_results(
    simulation_states: List[DigitalTwinState],
    scenario_results: Dict[str, Dict[str, Any]],
    digital_twin: IntegratedDigitalTwin,
) -> None:
    """Plot state metrics, forecast references, scenarios, and route allocations."""
    if not simulation_states:
        raise ValueError("At least one simulation state is required for visualization")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    ax1, ax2, ax3, ax4 = axes.flat

    timestamps = [state.timestamp for state in simulation_states]
    sync_health = [state.synchronization_health for state in simulation_states]
    service_levels = [
        state.performance_metrics["average_service_level"]
        for state in simulation_states
    ]
    ax1.plot(timestamps, sync_health, label="Synchronization-health proxy")
    ax1.plot(timestamps, service_levels, label="Mean route service level")
    ax1.set_xlabel("Simulated time")
    ax1.set_ylabel("Rate (0 to 1)")
    ax1.set_title("Recorded State Metrics")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis="x", rotation=30)

    sample_indices = np.linspace(
        0, len(simulation_states) - 1, min(10, len(simulation_states)), dtype=int
    )
    reference_demand = np.array(
        [route.demand_units for route in digital_twin.routes], dtype=float
    )
    forecasts = np.array(
        [simulation_states[index].demand_forecast for index in sample_indices]
    )
    reference_values = np.tile(reference_demand, len(sample_indices))
    forecast_values = forecasts.flatten()
    ax2.scatter(reference_values, forecast_values, alpha=0.7, label="Route-time samples")
    lower = float(min(reference_values.min(), forecast_values.min()))
    upper = float(max(reference_values.max(), forecast_values.max()))
    ax2.plot([lower, upper], [lower, upper], "r--", label="Equal-value reference")
    ax2.set_xlabel("Configured route demand (cargo units)")
    ax2.set_ylabel("Synthetic forecast (cargo units)")
    ax2.set_title("Forecasts and Configured Demand References")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    scenario_names = list(scenario_results)
    costs = [scenario_results[name]["total_cost"] for name in scenario_names]
    bars = ax3.bar(scenario_names, costs, edgecolor="black", alpha=0.8)
    ax3.set_ylabel("Combined modeled cost ($)")
    ax3.set_title("Scenario Cost Under the Stated Cost Model")
    ax3.tick_params(axis="x", rotation=35)
    ax3.ticklabel_format(style="plain", axis="y")
    ax3.grid(True, alpha=0.3, axis="y")
    for bar, cost in zip(bars, costs):
        ax3.annotate(
            f"${cost:,.0f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )

    final_allocation = simulation_states[-1].fleet_allocation
    route_capacities = np.zeros(len(digital_twin.routes), dtype=float)
    for vehicle_index, vehicle in enumerate(digital_twin.vehicles):
        route_capacities += final_allocation[vehicle_index] * vehicle.capacity
    final_demand = simulation_states[-1].demand_forecast
    route_service = np.array(
        [
            min(1.0, capacity / demand) if demand > 0 else 1.0
            for capacity, demand in zip(route_capacities, final_demand)
        ]
    )
    image = ax4.imshow(route_service.reshape(1, -1), cmap="YlOrRd", vmin=0, vmax=1)
    ax4.set_yticks([0])
    ax4.set_yticklabels(["Route service"])
    ax4.set_xticks(range(len(digital_twin.routes)))
    ax4.set_xticklabels([route.name for route in digital_twin.routes])
    ax4.set_title("Final Route Service Levels")
    for route_index, service in enumerate(route_service):
        ax4.text(route_index, 0, f"{service:.1%}", ha="center", va="center")
    colorbar = fig.colorbar(image, ax=ax4)
    colorbar.set_label("Service level (0 to 1)")

    fig.tight_layout()
    plt.show()


visualize_digital_twin_results(simulation_states, scenario_results, digital_twin)

In [ ]:
def generate_digital_twin_insights(
    simulation_states: List[DigitalTwinState],
    scenario_results: Dict[str, Dict[str, Any]],
    digital_twin: IntegratedDigitalTwin,
) -> List[str]:
    """Summarize recorded metrics without presenting proxies as measurements."""
    if not simulation_states:
        raise ValueError("At least one simulation state is required")

    insights: List[str] = []
    initial_state = simulation_states[0]
    final_state = simulation_states[-1]
    mean_sync_proxy = float(
        np.mean([state.synchronization_health for state in simulation_states])
    )
    initial_service = initial_state.performance_metrics["average_service_level"]
    final_service = final_state.performance_metrics["average_service_level"]
    service_change = 100 * (final_service - initial_service)

    insights.append(
        f"Mean synchronization-health proxy across recorded states: {mean_sync_proxy:.1%}."
    )
    insights.append(
        f"Mean route service level changed by {service_change:+.1f} percentage points "
        "from the first to the last recorded state."
    )
    insights.append(
        f"Final combined modeled cost: "
        f"${final_state.performance_metrics['total_cost']:,.2f}."
    )
    insights.append(
        f"Final fleet utilization, defined as capacity used to meet forecast demand: "
        f"{final_state.performance_metrics['fleet_utilization']:.1%}."
    )

    baseline_cost = scenario_results["baseline"]["total_cost"]
    for name, result in scenario_results.items():
        if name == "baseline":
            continue
        cost_change = result["total_cost"] - baseline_cost
        service_change_scenario = 100 * (
            result["average_service_level"]
            - scenario_results["baseline"]["average_service_level"]
        )
        insights.append(
            f"{name.replace('_', ' ').title()}: modeled cost change "
            f"${cost_change:+,.2f}; mean service-level change "
            f"{service_change_scenario:+.1f} percentage points."
        )

    sensor_health = digital_twin.sensor_network.get_sensor_health()
    insights.append(
        f"Configured sensor status: {sensor_health['active_sensors']}/"
        f"{sensor_health['total_sensors']} active; declared mean accuracy "
        f"{sensor_health['average_accuracy']:.1%}."
    )
    insights.append(
        "Forecast confidence is a simulated model proxy; no observed forecast-accuracy "
        "evaluation is available in this synthetic example."
    )
    return insights


insights = generate_digital_twin_insights(
    simulation_states, scenario_results, digital_twin
)
print("SIMULATION INSIGHTS")
print("=" * 60)
for insight in insights:
    print(f"- {insight}")

In [ ]:
if not simulation_states:
    raise ValueError("The simulation must record at least one state")

final_state = simulation_states[-1]
final_fleet: Dict[Tuple[int, int], int] = {}
for vehicle_index, vehicle in enumerate(vehicles):
    for route_index, route in enumerate(routes):
        count = int(final_state.fleet_allocation[vehicle_index, route_index])
        if count > 0:
            final_fleet[(vehicle.id, route.id)] = count

baseline_cost = scenario_results["baseline"]["total_cost"]
scenario_improvements = {
    name: 100 * (baseline_cost - result["total_cost"]) / baseline_cost
    for name, result in scenario_results.items()
    if name != "baseline" and baseline_cost != 0
}
mean_sync_proxy = float(
    np.mean([state.synchronization_health for state in simulation_states])
)
confidence_proxy = digital_twin.analytics.get_prediction_confidence()

digital_twin_result = DigitalTwinResult(
    final_fleet=final_fleet,
    total_cost=final_state.performance_metrics["total_cost"],
    synchronization_accuracy=mean_sync_proxy,
    prediction_accuracy=confidence_proxy,
    scenario_improvements=scenario_improvements,
    computation_time=float(len(simulation_states) * digital_twin.update_frequency),
    system_uptime=digital_twin.system_uptime,
    insights=insights,
)

assert np.all(final_state.fleet_allocation >= 0)
assert np.all(final_state.fleet_allocation == np.floor(final_state.fleet_allocation))
assert np.isclose(
    digital_twin_result.total_cost,
    final_state.performance_metrics["total_cost"],
)

print("FINAL SIMULATION SUMMARY")
print("=" * 60)
print("Final recorded allocation (not an optimality claim):")
for (vehicle_id, route_id), count in digital_twin_result.final_fleet.items():
    vehicle = next(vehicle for vehicle in vehicles if vehicle.id == vehicle_id)
    route = next(route for route in routes if route.id == route_id)
    print(f"  {count} x {vehicle.name} assigned to {route.name}")
print(f"Combined modeled cost: ${digital_twin_result.total_cost:,.2f}")
print(
    "Mean synchronization-health proxy: "
    f"{digital_twin_result.synchronization_accuracy:.1%}"
)
print(
    "Simulated demand-confidence proxy (not measured accuracy): "
    f"{digital_twin_result.prediction_accuracy:.1%}"
)
print(f"Configured system uptime assumption: {digital_twin_result.system_uptime:.1f}%")
print(f"Simulated horizon represented: {digital_twin_result.computation_time:.1f} seconds")
print("Scenario cost changes relative to the baseline:")
for name, change in digital_twin_result.scenario_improvements.items():
    print(f"  {name.replace('_', ' ').title()}: {change:+.1f}%")

In [ ]:
def summarize_model_capabilities(
    digital_twin_result: DigitalTwinResult,
    simulation_states: List[DigitalTwinState],
) -> None:
    """Report what this simulation represents and what it does not establish."""
    print("MODEL CAPABILITIES AND INTERPRETATION")
    print("=" * 60)
    print(f"Recorded simulation states: {len(simulation_states)}")
    print(f"Final combined modeled cost: ${digital_twin_result.total_cost:,.2f}")
    print("Included: synthetic sensor readings, synthetic forecasts, and state updates.")
    print("Included: a service-threshold rule and allocation-based scenario evaluation.")
    print("Not established: global optimality, measured forecast accuracy, or real synchronization.")
    print("Not modeled: fuel prices, vehicle purchase limits, emissions, and lifecycle costs.")
    print("The final fleet is the simulation outcome, not a solver-certified optimum.")


summarize_model_capabilities(digital_twin_result, simulation_states)

## 5. Interpreting this tier

The system-of-systems perspective connects fleet allocation with sensor data, route conditions, forecasts, state records, and scenario evaluation. Its educational value is in showing how these components can exchange information within one simulation workflow. The numerical readings are synthetic, so the results illustrate model behavior rather than operational performance.

### Capabilities represented

- **Virtual state:** the `DigitalTwinState` record groups allocation, mock readings, forecasts, conditions, and metrics at a simulated timestamp.
- **Sensor network:** sensor definitions produce synthetic GPS, fuel, load, maintenance, and traffic readings. The example contains four sensors per vehicle type and one per route.
- **Predictive analytics:** generated historical demand supports a simple trend-based forecast and a historical-standard-deviation anomaly flag. The confidence value is a proxy, not a validation statistic.
- **State synchronization proxy:** mock-reading coverage and declared sensor accuracy are combined into a descriptive score. Real physical-to-virtual synchronization would require time-aligned observations and measured data quality.
- **Scenario evaluation:** alternative allocations and a scaled demand forecast are evaluated with the same stated cost and service equations. A scenario can only quantify effects represented by those equations.
- **Threshold-based allocation adjustment:** the simulation can add a small truck when modeled service falls below its threshold. This rule is illustrative and does not establish cost optimality.

### Limitations and possible extensions

The model omits physical vehicle telemetry, stochastic travel and service times, vehicle availability, purchase limits, fuel prices, maintenance costs, emissions, and lifecycle accounting. Its acquisition-plus-operating cost combines different cost timing conventions for a compact demonstration. A practical study would need to define a consistent planning horizon, calibrate the demand process, validate forecasts against observed demand, and verify that all relevant operational constraints are represented. These additions would extend the model; they are not implemented here.

In [ ]:
print("TIER 5 SUMMARY: SYSTEM-OF-SYSTEMS DIGITAL-TWIN SIMULATION")
print("=" * 70)
print("The notebook assembled a synthetic fleet-management workflow with:")
print("  - Mock IoT readings for vehicle and route sensor types")
print("  - Synthetic demand-history generation and trend-based forecasts")
print("  - Timestamped virtual states and route-level capacity metrics")
print("  - A simple service-threshold allocation rule")
print("  - What-if scenarios evaluated under explicit model assumptions")
print()
print("Interpret results as simulation outputs, not measured operational evidence.")
print("The final allocation is not guaranteed to minimize the modeled cost.")
print("Forecast confidence and synchronization health are proxies, not validated metrics.")